In [0]:
%sql
CREATE TABLE IF NOT EXISTS dev.silver.products_scd2 (
    product_id INT,
    product_name STRING,
    category STRING,
    price DECIMAL(10,2),
    supplier_id INT,
    effective_date DATE,
    end_date DATE,
    is_current BOOLEAN,
    version INT
)


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW products_raw_view AS
SELECT product_id, product_name, category, coalesce(price, 0) as price, supplier_id,
       ingestion_date
FROM (
    SELECT *,
           row_number() OVER (PARTITION BY product_id ORDER BY ingestion_date DESC) AS rn
    FROM identifier(:catalog || '.bronze.products_raw')
    WHERE product_id IS NOT NULL
)
WHERE rn = 1;

In [0]:
%sql
MERGE INTO identifier(:catalog || '.silver.products_scd2') t
USING products_raw_view s
ON t.product_id = s.product_id
   AND t.is_current = true
   AND (
        t.product_name <> s.product_name
        OR t.category <> s.category
        OR t.price <> s.price
        OR t.supplier_id <> s.supplier_id
   )
WHEN MATCHED THEN UPDATE SET
    t.end_date = s.ingestion_date,
    t.is_current = false;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW products_with_version AS
SELECT 
    s.product_id,
    s.product_name,
    s.category,
    s.price,
    s.supplier_id,
    s.ingestion_date,
    COALESCE(MAX(t.version), 0) + 1 AS next_version
FROM products_raw_view s
LEFT JOIN identifier(:catalog || '.silver.products_scd2') t
    ON s.product_id = t.product_id
GROUP BY s.product_id, s.product_name, s.category, s.price, s.supplier_id, s.ingestion_date;

In [0]:
%sql
MERGE INTO identifier(:catalog || '.silver.products_scd2') t
USING products_with_version s
ON t.product_id = s.product_id AND t.is_current = true
WHEN NOT MATCHED THEN INSERT (
    product_id, product_name, category, price, supplier_id,
    effective_date, end_date, is_current, version
) VALUES (
    s.product_id, s.product_name, s.category, s.price, s.supplier_id,
    s.ingestion_date, NULL, true, s.next_version
);